# Models

The lower-level API: tokenizer, model, then `generate`. Weights load in 4-bit so a small GPU can hold them.

`HF_TOKEN` comes from the repo `.env`. These cells match `transformers==4.57.6` with `bitsandbytes` and `accelerate`. A CUDA GPU is required.

`LLAMA` is Llama 3.2 1B Instruct. Swap it for `meta-llama/Meta-Llama-3.1-8B-Instruct` if you already have access to the 8B model. Llama and Gemma are gated: [Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct), [gemma-3-270m-it](https://huggingface.co/google/gemma-3-270m-it).


## Setup

In [ ]:
import gc
import os
from pathlib import Path

import torch
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

hf_home = repo_root / "W3-Open Source Models" / "hf-cache"
hf_home.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_home)
os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_home / "datasets")

from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextStreamer

login(os.getenv("HF_TOKEN"))
print("Model cache:", hf_home)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU")

device = "cuda"
print(torch.cuda.get_device_name(0))

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

joke = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"},
]
light_joke = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"},
]

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)


## Llama input ids

This chat template does not add the generation prompt. The tensor is the whole input the model will see.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(joke, return_tensors="pt").to(device)
print(inputs)


## Layers

The printout is a PyTorch transformer: an embedding, a stack of decoder layers, and a language-model head. Each decoder layer has attention, an MLP, and norms. Llama 3.2 1B has 16 decoder layers. Llama 3.1 8B has 32. The weights below are 4-bit.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config,
)
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")
print(model)


## Raw generation

The first print is token ids. The second is those ids decoded. Without the generation prompt, the model continues the user turn instead of answering it.

In [ ]:
outputs = model.generate(inputs, max_new_tokens=80)
print(outputs[0])
print(tokenizer.decode(outputs[0]))


In [ ]:
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()


## Streamed generation

`add_generation_prompt=True` opens the assistant turn, so the model answers. `TextStreamer` prints tokens as they arrive. `quant=False` loads full weights, for the smaller models.

The generation prompt is described in the [chat templating docs](https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts).

In [ ]:
def generate(model_name, messages, quant=True, max_new_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(device)
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)
    streamer = TextStreamer(tokenizer)
    kwargs = {"device_map": "auto"}
    if quant:
        kwargs["quantization_config"] = quant_config
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        streamer=streamer,
    )
    del model, input_ids, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


## Phi

4-bit.

In [ ]:
generate(PHI, joke)


## Gemma

Full weights. This model is small, and it is gated.

In [ ]:
generate(GEMMA, light_joke, quant=False)


## Qwen

4-bit.

In [ ]:
generate(QWEN, light_joke)


## DeepSeek

Full weights. This is a reasoning distill, so it gets 500 new tokens.

In [ ]:
generate(DEEPSEEK, light_joke, quant=False, max_new_tokens=500)
